# Income Layer Preparation

## Objective

Prepare the HMRC Survey of Personal Incomes dataset for integration into the London Intelligence Dataset.

This notebook transforms the raw borough-level income data into a validated borough-year feature layer while preserving the original source values and maintaining a consistent merge strategy.

In [1]:
import pandas as pd

income_file = "../data/raw/borough_income_taxpayers.xlsx"

df_income = pd.read_excel(
    income_file,
    sheet_name="Total Income",
    header=[0, 1]
)

df_income.head()

Unnamed: 0_level_0    Unnamed: 1_level_0               1999-00            \
                Code                  Area Number of Individuals    Mean £   
0                NaN                   NaN                   NaN       NaN   
1          E09000001        City of London               10000.0  109800.0   
2          E09000002  Barking and Dagenham               62000.0   16200.0   
3          E09000003                Barnet              161000.0   26800.0   
4          E09000004                Bexley              105000.0   20500.0   

                         2000-01                                  2001-02  \
  Median £ Number of Individuals    Mean £ Median £ Number of Individuals   
0      NaN                   NaN       NaN      NaN                   NaN   
1  40400.0               10000.0  137000.0  65000.0               12000.0   
2  15100.0               71000.0   18100.0  15600.0               56000.0   
3  18700.0              156000.0   30800.0  19400.0              159000.0   
4  17200.0              116000.0   19800.0  17300.0              112000.0   

            ...  2020-21               2021-22                     \
    Mean £  ... Median £ Number of Individuals    Mean £ Median £   
0      NaN  ...      NaN                   NaN       NaN      NaN   
1  92900.0  ...  56700.0                9000.0  167000.0  62200.0   
2  18600.0  ...  26800.0               90000.0   31800.0  28000.0   
3  30400.0  ...  32200.0              190000.0   59100.0  33500.0   
4  22000.0  ...  28800.0              135000.0   38000.0  29300.0   

                2022-23                                  2023-24            \
  Number of Individuals    Mean £ Median £ Number of Individuals    Mean £   
0                   NaN       NaN      NaN                   NaN       NaN   
1               11000.0  147000.0  60700.0               11000.0  162000.0   
2               98000.0   33300.0  28600.0              106000.0   34600.0   
3              198000.0   59800.0  34400.0              216000.0   61600.0   
4              134000.0   40900.0  31800.0              142000.0   42600.0   

            
  Median £  
0      NaN  
1  70000.0  
2  29400.0  
3  36100.0  
4  33200.0  

[5 rows x 74 columns]

In [2]:
print("Shape:", df_income.shape)

print("\nColumns:")
print(df_income.columns)

print("\nData Types:")
print(df_income.dtypes)

Shape: (51, 74)

Columns:
MultiIndex([('Unnamed: 0_level_0',                  'Code'),
            ('Unnamed: 1_level_0',                  'Area'),
            (           '1999-00', 'Number of Individuals'),
            (           '1999-00',                'Mean £'),
            (           '1999-00',              'Median £'),
            (           '2000-01', 'Number of Individuals'),
            (           '2000-01',                'Mean £'),
            (           '2000-01',              'Median £'),
            (           '2001-02', 'Number of Individuals'),
            (           '2001-02',                'Mean £'),
            (           '2001-02',              'Median £'),
            (           '2002-03', 'Number of Individuals'),
            (           '2002-03',                'Mean £'),
            (           '2002-03',              'Median £'),
            (           '2003-04', 'Number of Individuals'),
            (           '2003-04',                'Mean £')

In [3]:
print("First 10 Code / Area records:")
display(df_income.iloc[:, :2].head(10))

print("\nUnique Codes:")
print(df_income.iloc[:, 0].dropna().unique())

print("\nUnique Areas:")
print(sorted(df_income.iloc[:, 1].dropna().unique()))

First 10 Code / Area records:


,Unnamed: 0_level_0,Unnamed: 1_level_0
,Code,Area
0,NaN,NaN
1,E09000001,City of London
2,E09000002,Barking and Dagenham
3,E09000003,Barnet
4,E09000004,Bexley
5,E09000005,Brent
6,E09000006,Bromley
7,E09000007,Camden
8,E09000008,Croydon



Unique Codes:
['E09000001' 'E09000002' 'E09000003' 'E09000004' 'E09000005' 'E09000006'
 'E09000007' 'E09000008' 'E09000009' 'E09000010' 'E09000011' 'E09000012'
 'E09000013' 'E09000014' 'E09000015' 'E09000016' 'E09000017' 'E09000018'
 'E09000019' 'E09000020' 'E09000021' 'E09000022' 'E09000023' 'E09000024'
 'E09000025' 'E09000026' 'E09000027' 'E09000028' 'E09000029' 'E09000030'
 'E09000031' 'E09000032' 'E09000033' 'E12000001' 'E12000002' 'E12000003'
 'E12000004' 'E12000005' 'E12000006' 'E12000007' 'E12000008' 'E12000009'
 'E92000001' 'W92000004' 'S92000003' 'N92000002' 'K02000001']

Unique Areas:
['Barking and Dagenham', 'Barnet', 'Bexley', 'Brent', 'Bromley', 'Camden', 'City of London', 'Croydon', 'Ealing', 'East Midlands', 'East of England', 'Enfield', 'England', 'Greenwich', 'Hackney', 'Hammersmith and Fulham', 'Haringey', 'Harrow', 'Havering', 'Hillingdon', 'Hounslow', 'Islington', 'Kensington and Chelsea', 'Kingston-upon-Thames', 'Lambeth', 'Lewisham', 'London', 'Merton', 'Newham',

In [4]:
# Inspect all geographical records

area_series = df_income.iloc[:, 1]

print("Number of non-null areas:", area_series.notna().sum())
print("Number of unique areas:", area_series.dropna().nunique())

print("\nAll areas:")
for area in area_series.dropna():
    print(area)

Number of non-null areas: 47
Number of unique areas: 47

All areas:
City of London
Barking and Dagenham
Barnet
Bexley
Brent
Bromley
Camden
Croydon
Ealing
Enfield
Greenwich
Hackney
Hammersmith and Fulham
Haringey
Harrow
Havering
Hillingdon
Hounslow
Islington
Kensington and Chelsea
Kingston-upon-Thames
Lambeth
Lewisham
Merton
Newham
Redbridge
Richmond-upon-Thames
Southwark
Sutton
Tower Hamlets
Waltham Forest
Wandsworth
Westminster
North East
North West
Yorkshire and The Humber
East Midlands
West Midlands
East of England
London
South East
South West
England
Wales
Scotland
Northern Ireland
United Kingdom


In [5]:
# Inspect all geographical codes

code_series = df_income.iloc[:, 0]

print("Number of non-null codes:", code_series.notna().sum())
print("Number of unique codes:", code_series.dropna().nunique())

print("\nAll codes:")
for code in code_series.dropna():
    print(code)

Number of non-null codes: 47
Number of unique codes: 47

All codes:
E09000001
E09000002
E09000003
E09000004
E09000005
E09000006
E09000007
E09000008
E09000009
E09000010
E09000011
E09000012
E09000013
E09000014
E09000015
E09000016
E09000017
E09000018
E09000019
E09000020
E09000021
E09000022
E09000023
E09000024
E09000025
E09000026
E09000027
E09000028
E09000029
E09000030
E09000031
E09000032
E09000033
E12000001
E12000002
E12000003
E12000004
E12000005
E12000006
E12000007
E12000008
E12000009
E92000001
W92000004
S92000003
N92000002
K02000001


In [6]:
# Identify London local authorities using official HMRC codes

london_codes = [
    f"E090000{i:02d}"
    for i in range(1, 34)
]

df_income_london = df_income[
    df_income.iloc[:, 0].isin(london_codes)
].copy()

print("London records:", len(df_income_london))
print("Unique London codes:", df_income_london.iloc[:, 0].nunique())
print("Unique London areas:", df_income_london.iloc[:, 1].nunique())

print("\nLondon areas:")
display(df_income_london.iloc[:, :2])

London records: 33
Unique London codes: 33
Unique London areas: 33

London areas:


,Unnamed: 0_level_0,Unnamed: 1_level_0
,Code,Area
1,E09000001,City of London
2,E09000002,Barking and Dagenham
3,E09000003,Barnet
4,E09000004,Bexley
5,E09000005,Brent
6,E09000006,Bromley
7,E09000007,Camden
8,E09000008,Croydon
9,E09000009,Ealing


In [7]:
# Check London code sequence

expected_codes = set(london_codes)
actual_codes = set(df_income_london.iloc[:, 0])

print("Missing London codes:", sorted(expected_codes - actual_codes))
print("Unexpected London codes:", sorted(actual_codes - expected_codes))

Missing London codes: []
Unexpected London codes: []


## Clean Column Structure

Standardise the Code and Area columns while preserving the original source values and the annual income data structure.

In [8]:
# Clean the first two column levels

df_income_london = df_income_london.copy()

df_income_london.columns = pd.MultiIndex.from_tuples(
    [
        ("", "Code") if col[1] == "Code"
        else ("", "Area") if col[1] == "Area"
        else col
        for col in df_income_london.columns
    ]
)

df_income_london.head()

1999-00                     \
        Code                  Area Number of Individuals    Mean £ Median £   
1  E09000001        City of London               10000.0  109800.0  40400.0   
2  E09000002  Barking and Dagenham               62000.0   16200.0  15100.0   
3  E09000003                Barnet              161000.0   26800.0  18700.0   
4  E09000004                Bexley              105000.0   20500.0  17200.0   
5  E09000005                 Brent              122000.0   20000.0  16500.0   

                2000-01                                  2001-02           \
  Number of Individuals    Mean £ Median £ Number of Individuals   Mean £   
1               10000.0  137000.0  65000.0               12000.0  92900.0   
2               71000.0   18100.0  15600.0               56000.0  18600.0   
3              156000.0   30800.0  19400.0              159000.0  30400.0   
4              116000.0   19800.0  17300.0              112000.0  22000.0   
5              130000.0   22200.0  16800.0              127000.0  21100.0   

   ...  2020-21               2021-22                     \
   ... Median £ Number of Individuals    Mean £ Median £   
1  ...  56700.0                9000.0  167000.0  62200.0   
2  ...  26800.0               90000.0   31800.0  28000.0   
3  ...  32200.0              190000.0   59100.0  33500.0   
4  ...  28800.0              135000.0   38000.0  29300.0   
5  ...  27200.0              149000.0   44100.0  30000.0   

                2022-23                                  2023-24            \
  Number of Individuals    Mean £ Median £ Number of Individuals    Mean £   
1               11000.0  147000.0  60700.0               11000.0  162000.0   
2               98000.0   33300.0  28600.0              106000.0   34600.0   
3              198000.0   59800.0  34400.0              216000.0   61600.0   
4              134000.0   40900.0  31800.0              142000.0   42600.0   
5              170000.0   43600.0  29400.0              182000.0   44900.0   

            
  Median £  
1  70000.0  
2  29400.0  
3  36100.0  
4  33200.0  
5  30100.0  

[5 rows x 74 columns]

In [9]:
print(df_income_london.columns[:8])

MultiIndex([(       '',                  'Code'),
            (       '',                  'Area'),
            ('1999-00', 'Number of Individuals'),
            ('1999-00',                'Mean £'),
            ('1999-00',              'Median £'),
            ('2000-01', 'Number of Individuals'),
            ('2000-01',                'Mean £'),
            ('2000-01',              'Median £')],
           )


In [10]:
# Identify available tax years

income_years = sorted(
    {
        col[0]
        for col in df_income_london.columns
        if col[0] != ""
    }
)

print("Number of tax years:", len(income_years))
print("\nTax years:")
print(income_years)

Number of tax years: 24

Tax years:
['1999-00', '2000-01', '2001-02', '2002-03', '2003-04', '2004-05', '2005-06', '2006-07', '2007-08', '2009-10', '2010-11', '2011-12', '2012-13', '2013-14', '2014-15', '2015-16', '2016-17', '2017-18', '2018-19', '2019-20', '2020-21', '2021-22', '2022-23', '2023-24']


### Audit Result

- The dataset contains 24 tax years, covering 1999-00 to 2023-24.
- The 2008-09 tax year is absent from the source data.
- This gap is recorded for later consideration and has not been filled or inferred.
- Source values remain unchanged at this stage.

In [11]:
# Audit missing values across London boroughs and tax years

income_data_columns = [
    col
    for col in df_income_london.columns
    if col[0] != ""
]

missing_by_year = df_income_london[income_data_columns].isna().sum()

print("Missing values by tax year:")
display(missing_by_year)

Missing values by tax year:


1999-00  Number of Individuals    0
         Mean £                   0
         Median £                 0
2000-01  Number of Individuals    0
         Mean £                   0
                                 ..
2022-23  Mean £                   0
         Median £                 0
2023-24  Number of Individuals    0
         Mean £                   0
         Median £                 0
Length: 72, dtype: int64

### Audit Result

- No missing values were found across the 33 London boroughs.
- All 24 available tax years contain complete values for all three income measures.
- The resulting 72 annual data columns (`24 × 3`) are fully populated.
- No missing values have been imputed or otherwise modified.

In [12]:
# Audit data types of annual income measures

income_measure_dtypes = {
    measure: [
        df_income_london[(year, measure)].dtype
        for year in income_years
    ]
    for measure in [
        "Number of Individuals",
        "Mean £",
        "Median £"
    ]
}

for measure, dtypes in income_measure_dtypes.items():
    print(f"{measure}:")
    print(set(dtypes))
    print()

Number of Individuals:
{dtype('float64')}

Mean £:
{dtype('float64')}

Median £:
{dtype('float64')}



### Audit Result

- All three annual income measures use a consistent numeric data type (`float64`) across all 24 available tax years.
- No data type inconsistencies were identified.
- The source data types are preserved at this stage; no type conversion has been applied.

In [13]:
# Audit value ranges and unexpected values

income_measures = [
    "Number of Individuals",
    "Mean £",
    "Median £"
]

for measure in income_measures:
    values = pd.concat(
        [
            df_income_london[(year, measure)]
            for year in income_years
        ],
        ignore_index=True
    )

    print(f"{measure}")
    print(f"  Minimum: {values.min():,.0f}")
    print(f"  Maximum: {values.max():,.0f}")
    print(f"  Negative values: {(values < 0).sum()}")
    print(f"  Zero values: {(values == 0).sum()}")
    print()

Number of Individuals
  Minimum: 8,000
  Maximum: 220,000
  Negative values: 0
  Zero values: 0

Mean £
  Minimum: 15,800
  Maximum: 209,000
  Negative values: 0
  Zero values: 0

Median £
  Minimum: 13,300
  Maximum: 70,000
  Negative values: 0
  Zero values: 0



### Audit Result

- No negative or zero values were found across the three annual income measures.
- `Number of Individuals` ranges from 8,000 to 220,000.
- `Mean £` ranges from £15,800 to £209,000.
- `Median £` ranges from £13,300 to £70,000.
- No immediate value-range anomalies were identified.
- No values have been modified or removed at this stage.

In [14]:
# Audit consistency between mean and median income

mean_median_issues = []

for year in income_years:
    mean_values = df_income_london[(year, "Mean £")]
    median_values = df_income_london[(year, "Median £")]

    issues = df_income_london.loc[
        mean_values < median_values,
        [("", "Code"), ("", "Area")]
    ].copy()

    if not issues.empty:
        issues["Tax Year"] = year
        issues["Mean £"] = mean_values[mean_values < median_values].values
        issues["Median £"] = median_values[mean_values < median_values].values

        mean_median_issues.append(issues)

if mean_median_issues:
    mean_median_issues = pd.concat(mean_median_issues, ignore_index=True)
else:
    mean_median_issues = pd.DataFrame()

print("Number of Mean < Median cases:", len(mean_median_issues))

if not mean_median_issues.empty:
    display(mean_median_issues)

Number of Mean < Median cases: 0


### Audit Result

- Mean income is greater than or equal to median income for all 792 borough-year observations.
- No `Mean £ < Median £` inconsistencies were identified.
- The relationship between mean and median income shows no immediate structural anomaly.
- No source values have been modified.

In [15]:
# Audit borough-year coverage

expected_boroughs = 33
expected_years = len(income_years)
expected_observations = expected_boroughs * expected_years

actual_boroughs = df_income_london.iloc[:, 0].nunique()
actual_years = len(income_years)

print("Expected boroughs:", expected_boroughs)
print("Actual boroughs:", actual_boroughs)

print("\nExpected tax years:", expected_years)
print("Actual tax years:", actual_years)

print("\nExpected borough-year observations:", expected_observations)
print("Actual borough-year observations:", len(df_income_london) * actual_years)

Expected boroughs: 33
Actual boroughs: 33

Expected tax years: 24
Actual tax years: 24

Expected borough-year observations: 792
Actual borough-year observations: 792


### Audit Result

- All 33 London boroughs are present in the dataset.
- All 24 available tax years are represented.
- The dataset contains the expected 792 borough-year observations (`33 × 24`).
- No structural gaps were identified in the available borough-year coverage.

In [16]:
# Audit income measures across all tax years

expected_income_measures = {
    "Number of Individuals",
    "Mean £",
    "Median £"
}

measures_by_year = {
    year: {
        col[1]
        for col in df_income_london.columns
        if col[0] == year
    }
    for year in income_years
}

for year, measures in measures_by_year.items():
    print(f"{year}: {sorted(measures)}")

unexpected_measures = {
    measure
    for measures in measures_by_year.values()
    for measure in measures
    if measure not in expected_income_measures
}

missing_measures = {
    year: expected_income_measures - measures
    for year, measures in measures_by_year.items()
    if expected_income_measures - measures
}

print("\nUnexpected measures:", unexpected_measures)
print("Years with missing measures:", missing_measures)

1999-00: ['Mean £', 'Median £', 'Number of Individuals']
2000-01: ['Mean £', 'Median £', 'Number of Individuals']
2001-02: ['Mean £', 'Median £', 'Number of Individuals']
2002-03: ['Mean £', 'Median £', 'Number of Individuals']
2003-04: ['Mean £', 'Median £', 'Number of Individuals']
2004-05: ['Mean £', 'Median £', 'Number of Individuals']
2005-06: ['Mean £', 'Median £', 'Number of Individuals']
2006-07: ['Mean £', 'Median £', 'Number of Individuals']
2007-08: ['Mean £', 'Median £', 'Number of Individuals']
2009-10: ['Mean £', 'Median £', 'Number of Individuals']
2010-11: ['Mean £', 'Median £', 'Number of Individuals']
2011-12: ['Mean £', 'Median £', 'Number of Individuals']
2012-13: ['Mean £', 'Median £', 'Number of Individuals']
2013-14: ['Mean £', 'Median £', 'Number of Individuals']
2014-15: ['Mean £', 'Median £', 'Number of Individuals']
2015-16: ['Mean £', 'Median £', 'Number of Individuals']
2016-17: ['Mean £', 'Median £', 'Number of Individuals']
2017-18: ['Mean £', 'Median £',

### Audit Result

- All 24 available tax years contain the same three income measures.
- The expected measures are `Number of Individuals`, `Mean £`, and `Median £`.
- No unexpected measures were identified.
- No tax year has missing income measures.
- The annual measure structure is consistent across the dataset.

In [17]:
# Inspect available worksheets in the source workbook

excel_file = pd.ExcelFile(income_file)

print("Available worksheets:")
print(excel_file.sheet_names)

Available worksheets:
['Metadata', 'Total Income']


In [18]:
# Inspect the source metadata

df_income_metadata = pd.read_excel(
    income_file,
    sheet_name="Metadata",
    header=None
)

print("Metadata shape:", df_income_metadata.shape)

display(df_income_metadata.head(20))

Metadata shape: (30, 4)


,0,1,2,3
0,Name,HMRC Survey of Personal Incomes,NaN,NaN
1,ShortName,SPI,NaN,NaN
2,NaN,NaN,NaN,NaN
3,NaN,NaN,NaN,NaN
4,NaN,NaN,NaN,NaN
5,Theme,Employment and Skills,NaN,NaN
6,Sub-theme,Income and earnings,NaN,NaN
7,NaN,NaN,NaN,NaN
8,Title,Average Income of Tax Payers,NaN,NaN
9,Description,Mean and Median Income (Personal incomes by ta...,NaN,NaN


In [19]:
# Display complete source metadata

pd.set_option("display.max_colwidth", None)

display(df_income_metadata)

,0,1,2,3
0,Name,HMRC Survey of Personal Incomes,NaN,NaN
1,ShortName,SPI,NaN,NaN
2,NaN,NaN,NaN,NaN
3,NaN,NaN,NaN,NaN
4,NaN,NaN,NaN,NaN
5,Theme,Employment and Skills,NaN,NaN
6,Sub-theme,Income and earnings,NaN,NaN
7,NaN,NaN,NaN,NaN
8,Title,Average Income of Tax Payers,NaN,NaN
9,Description,Mean and Median Income (Personal incomes by tax year),NaN,NaN


### Source Semantics

- Source: HMRC Survey of Personal Incomes (SPI).
- The dataset provides annual personal income measures by tax year for Local Authorities.
- The income measures include `Number of Individuals`, `Mean £`, and `Median £`.
- The source metadata confirms that measurements are reported as numbers and GBP.
- The metadata explicitly confirms that no data are available for tax year `2008-09`.
- HMRC notes that estimates for sub-UK geographical areas should be treated with particular caution.
- Source definitions and values are preserved without modification at this stage.

In [20]:
# Inspect tax-year labels before calendar-year mapping

tax_year_mapping_check = pd.DataFrame({
    "Tax_Year": income_years
})

display(tax_year_mapping_check)

,Tax_Year
0,1999-00
1,2000-01
2,2001-02
3,2002-03
4,2003-04
5,2004-05
6,2005-06
7,2006-07
8,2007-08
9,2009-10


## Identify Income Years Relevant to the Project

Identify the HMRC tax years that overlap with the calendar-year coverage of the London Intelligence Dataset.

In [21]:
# Identify tax years relevant to the project

project_years = set(range(2018, 2026))

income_tax_years_relevant = [
    tax_year
    for tax_year in income_years
    if int(tax_year[:4]) in project_years
]

print("Project years:", sorted(project_years))
print("\nRelevant Income tax years:")
print(income_tax_years_relevant)

Project years: [2018, 2019, 2020, 2021, 2022, 2023, 2024, 2025]

Relevant Income tax years:
['2018-19', '2019-20', '2020-21', '2021-22', '2022-23', '2023-24']


## Map HMRC Tax Years to Project Calendar Years

HMRC income data is reported by tax year, while the London Intelligence Dataset uses calendar years.

For integration, each HMRC tax year is mapped to its starting calendar year.

In [22]:
# Create an explicit Tax Year → Project Year mapping

tax_year_mapping = pd.DataFrame({
    "Tax_Year": income_tax_years_relevant,
    "Year": [int(tax_year[:4]) for tax_year in income_tax_years_relevant]
})

display(tax_year_mapping)

,Tax_Year,Year
0,2018-19,2018
1,2019-20,2019
2,2020-21,2020
3,2021-22,2021
4,2022-23,2022
5,2023-24,2023


## Validate Tax Year Mapping

Validate that each relevant HMRC tax year maps to exactly one project calendar year.

In [23]:
# Validate Tax Year → Project Year mapping

print("Number of mappings:", len(tax_year_mapping))

print("\nUnique Tax Years:", tax_year_mapping["Tax_Year"].nunique())
print("Unique Project Years:", tax_year_mapping["Year"].nunique())

print("\nDuplicate Tax Year mappings:")
print(tax_year_mapping["Tax_Year"].duplicated().sum())

print("\nDuplicate Project Year mappings:")
print(tax_year_mapping["Year"].duplicated().sum())

Number of mappings: 6

Unique Tax Years: 6
Unique Project Years: 6

Duplicate Tax Year mappings:
0

Duplicate Project Year mappings:
0


## Select Relevant Income Years

Restrict the Income dataset to the tax years that overlap with the London Intelligence Dataset.

In [24]:
# Select relevant tax years only

income_relevant_columns = [
    col
    for col in df_income_london.columns
    if col[0] in income_tax_years_relevant
]

# Keep Code and Area columns as well
income_relevant_columns = [
    df_income_london.columns[0],
    df_income_london.columns[1],
] + income_relevant_columns

df_income_relevant = df_income_london[
    income_relevant_columns
].copy()

print("Shape:", df_income_relevant.shape)

Shape: (33, 20)


## Reshape Income Data to Long Format

Convert the selected HMRC income data from wide format to long format.

Each row will represent one borough, one tax year and one income measure.

In [25]:
# Reshape Income data from wide to long format

income_long = (
    df_income_relevant
    .set_index([
        ("", "Code"),
        ("", "Area")
    ])
    .stack(level=[0, 1], future_stack=True)
    .reset_index()
)

# Rename columns
income_long.columns = [
    "Code",
    "Area",
    "Tax_Year",
    "Measure",
    "Value"
]

print("Shape:", income_long.shape)

display(income_long.head(10))

Shape: (594, 5)


,Code,Area,Tax_Year,Measure,Value
0,E09000001,City of London,2018-19,Number of Individuals,12000.0
1,E09000001,City of London,2018-19,Mean £,142000.0
2,E09000001,City of London,2018-19,Median £,55200.0
3,E09000001,City of London,2019-20,Number of Individuals,12000.0
4,E09000001,City of London,2019-20,Mean £,128000.0
5,E09000001,City of London,2019-20,Median £,54200.0
6,E09000001,City of London,2020-21,Number of Individuals,10000.0
7,E09000001,City of London,2020-21,Mean £,142000.0
8,E09000001,City of London,2020-21,Median £,56700.0
9,E09000001,City of London,2021-22,Number of Individuals,9000.0


## Validate Long-Format Income Data

Validate borough, tax-year and measure coverage after reshaping the Income dataset.

In [26]:
# Validate long-format structure

print("Unique boroughs:", income_long["Code"].nunique())
print("Unique tax years:", income_long["Tax_Year"].nunique())
print("Unique measures:", income_long["Measure"].nunique())

print("\nMeasures:")
print(sorted(income_long["Measure"].unique()))

print("\nExpected observations:", 33 * 6 * 3)
print("Actual observations:", len(income_long))

print("\nDuplicate Code-Tax_Year-Measure records:")
print(
    income_long.duplicated(
        subset=["Code", "Tax_Year", "Measure"]
    ).sum()
)

Unique boroughs: 33
Unique tax years: 6
Unique measures: 3

Measures:
['Mean £', 'Median £', 'Number of Individuals']

Expected observations: 594
Actual observations: 594

Duplicate Code-Tax_Year-Measure records:
0


## Create Borough-Year Income Features

Reshape the long-format Income data so that each borough-year observation contains separate features for taxpayer count, mean income and median income.

In [27]:
# Convert income measures into separate features

income_features = (
    income_long
    .pivot(
        index=["Code", "Area", "Tax_Year"],
        columns="Measure",
        values="Value"
    )
    .reset_index()
)

# Remove the columns index name
income_features.columns.name = None

print("Shape:", income_features.shape)

display(income_features.head())

Shape: (198, 6)


,Code,Area,Tax_Year,Mean £,Median £,Number of Individuals
0,E09000001,City of London,2018-19,142000.0,55200.0,12000.0
1,E09000001,City of London,2019-20,128000.0,54200.0,12000.0
2,E09000001,City of London,2020-21,142000.0,56700.0,10000.0
3,E09000001,City of London,2021-22,167000.0,62200.0,9000.0
4,E09000001,City of London,2022-23,147000.0,60700.0,11000.0


## Standardise Income Feature Names

Rename the HMRC income measures using clear, machine-readable feature names while preserving the original values.

In [28]:
# Rename income features

income_features = income_features.rename(
    columns={
        "Mean £": "Mean_Income",
        "Median £": "Median_Income",
        "Number of Individuals": "Number_of_Individuals"
    }
)

print("Columns:")
print(income_features.columns.tolist())

display(income_features.head())

Columns:
['Code', 'Area', 'Tax_Year', 'Mean_Income', 'Median_Income', 'Number_of_Individuals']


,Code,Area,Tax_Year,Mean_Income,Median_Income,Number_of_Individuals
0,E09000001,City of London,2018-19,142000.0,55200.0,12000.0
1,E09000001,City of London,2019-20,128000.0,54200.0,12000.0
2,E09000001,City of London,2020-21,142000.0,56700.0,10000.0
3,E09000001,City of London,2021-22,167000.0,62200.0,9000.0
4,E09000001,City of London,2022-23,147000.0,60700.0,11000.0


## Map Tax Years to Project Years

Map HMRC tax years to the corresponding project calendar years using the validated tax-year mapping.

In [29]:
# Map HMRC tax years to project years

income_features = income_features.merge(
    tax_year_mapping,
    on="Tax_Year",
    how="left",
    validate="many_to_one"
)

print("Shape:", income_features.shape)

print("\nYears:")
print(sorted(income_features["Year"].unique()))

display(income_features.head())

Shape: (198, 7)

Years:
[np.int64(2018), np.int64(2019), np.int64(2020), np.int64(2021), np.int64(2022), np.int64(2023)]


,Code,Area,Tax_Year,Mean_Income,Median_Income,Number_of_Individuals,Year
0,E09000001,City of London,2018-19,142000.0,55200.0,12000.0,2018
1,E09000001,City of London,2019-20,128000.0,54200.0,12000.0,2019
2,E09000001,City of London,2020-21,142000.0,56700.0,10000.0,2020
3,E09000001,City of London,2021-22,167000.0,62200.0,9000.0,2021
4,E09000001,City of London,2022-23,147000.0,60700.0,11000.0,2022


## Validate Tax Year Mapping

Validate the completeness and uniqueness of the tax-year to project-year mapping after integration into the Income feature table.

In [30]:
# Validate tax-year mapping

print("Missing Tax_Year values:", income_features["Tax_Year"].isna().sum())
print("Missing Year values:", income_features["Year"].isna().sum())

print("\nUnique Tax Years:", income_features["Tax_Year"].nunique())
print("Unique Project Years:", income_features["Year"].nunique())

print("\nTax Year → Project Year mapping:")
display(
    income_features[
        ["Tax_Year", "Year"]
    ]
    .drop_duplicates()
    .sort_values("Tax_Year")
)

Missing Tax_Year values: 0
Missing Year values: 0

Unique Tax Years: 6
Unique Project Years: 6

Tax Year → Project Year mapping:


,Tax_Year,Year
0,2018-19,2018
1,2019-20,2019
2,2020-21,2020
3,2021-22,2021
4,2022-23,2022
5,2023-24,2023


## Create Merge Key

Create a standardised borough key for cross-layer integration while preserving the original HMRC Area values.

In [31]:
# Create a standardised merge key

income_features["Merge_Key"] = (
    income_features["Area"]
    .str.upper()
    .str.replace("-", " ", regex=False)
)

display(
    income_features[
        ["Area", "Merge_Key"]
    ]
    .drop_duplicates()
    .sort_values("Area")
)

,Area,Merge_Key
6,Barking and Dagenham,BARKING AND DAGENHAM
12,Barnet,BARNET
18,Bexley,BEXLEY
24,Brent,BRENT
30,Bromley,BROMLEY
36,Camden,CAMDEN
0,City of London,CITY OF LONDON
42,Croydon,CROYDON
48,Ealing,EALING
54,Enfield,ENFIELD


In [32]:
# Audit Income Merge Key

print("Income records:", len(income_features))
print("Unique boroughs:", income_features["Merge_Key"].nunique())
print("Missing Merge_Key:", income_features["Merge_Key"].isna().sum())

print("\nDuplicate Borough-Year records:")
print(
    income_features.duplicated(
        subset=["Merge_Key", "Year"]
    ).sum()
)

Income records: 198
Unique boroughs: 33
Missing Merge_Key: 0

Duplicate Borough-Year records:
0


## Final Income Layer Audit

Perform a final structural and data-quality audit of the prepared Income Layer before integration.

In [33]:
# -----------------------------
# Final Income Layer Audit
# -----------------------------

print("=" * 60)
print("INCOME LAYER FINAL AUDIT")
print("=" * 60)

print(f"Shape: {income_features.shape}")

print("\nColumns:")
print(income_features.columns.tolist())

print("\nProject Years:")
print(sorted(income_features["Year"].unique()))

print("\nUnique Boroughs:")
print(income_features["Merge_Key"].nunique())

print("\nDuplicate Borough-Year records:")
print(
    income_features.duplicated(
        subset=["Merge_Key", "Year"]
    ).sum()
)

print("\nMissing Values:")
print(
    income_features[
        [
            "Code",
            "Area",
            "Tax_Year",
            "Year",
            "Mean_Income",
            "Median_Income",
            "Number_of_Individuals",
            "Merge_Key"
        ]
    ].isna().sum()
)

INCOME LAYER FINAL AUDIT
Shape: (198, 8)

Columns:
['Code', 'Area', 'Tax_Year', 'Mean_Income', 'Median_Income', 'Number_of_Individuals', 'Year', 'Merge_Key']

Project Years:
[np.int64(2018), np.int64(2019), np.int64(2020), np.int64(2021), np.int64(2022), np.int64(2023)]

Unique Boroughs:
33

Duplicate Borough-Year records:
0

Missing Values:
Code                     0
Area                     0
Tax_Year                 0
Year                     0
Mean_Income              0
Median_Income            0
Number_of_Individuals    0
Merge_Key                0
dtype: int64


## Validate Income Feature Values

Check the numerical integrity of the final Income features, including missing, zero and negative values, and the expected relationship between mean and median income.

In [34]:
# Validate numerical income features

income_columns = [
    "Mean_Income",
    "Median_Income",
    "Number_of_Individuals"
]

print("Negative values:")
print(
    (income_features[income_columns] < 0).sum()
)

print("\nZero values:")
print(
    (income_features[income_columns] == 0).sum()
)

print("\nMinimum values:")
print(
    income_features[income_columns].min()
)

print("\nMaximum values:")
print(
    income_features[income_columns].max()
)

print("\nMean < Median cases:")
print(
    (
        income_features["Mean_Income"]
        < income_features["Median_Income"]
    ).sum()
)

Negative values:
Mean_Income              0
Median_Income            0
Number_of_Individuals    0
dtype: int64

Zero values:
Mean_Income              0
Median_Income            0
Number_of_Individuals    0
dtype: int64

Minimum values:
Mean_Income              28800.0
Median_Income            24300.0
Number_of_Individuals     9000.0
dtype: float64

Maximum values:
Mean_Income              209000.0
Median_Income             70000.0
Number_of_Individuals    220000.0
dtype: float64

Mean < Median cases:
0


## Validate Borough-Year Coverage

Confirm that every London borough has exactly one observation for each available project year.

In [35]:
# Validate borough-year coverage

borough_year_counts = (
    income_features
    .groupby("Merge_Key")["Year"]
    .nunique()
)

print("Minimum years per borough:", borough_year_counts.min())
print("Maximum years per borough:", borough_year_counts.max())

print("\nBoroughs with incomplete year coverage:")
display(
    borough_year_counts[
        borough_year_counts != len(income_features["Year"].unique())
    ]
)

Minimum years per borough: 6
Maximum years per borough: 6

Boroughs with incomplete year coverage:


Series([], Name: Year, dtype: int64)

## Validate Income Layer Grain

Confirm that the Income Layer has one unique observation per borough and project year, using both the official HMRC Code and the standardised Merge_Key.

In [36]:
# Validate Income Layer grain

print("Duplicate Code-Year records:")
print(
    income_features.duplicated(
        subset=["Code", "Year"]
    ).sum()
)

print("\nDuplicate Merge_Key-Year records:")
print(
    income_features.duplicated(
        subset=["Merge_Key", "Year"]
    ).sum()
)

print("\nExpected borough-year observations:")
print(
    income_features["Merge_Key"].nunique()
    * income_features["Year"].nunique()
)

print("\nActual observations:")
print(len(income_features))

Duplicate Code-Year records:
0

Duplicate Merge_Key-Year records:
0

Expected borough-year observations:
198

Actual observations:
198


## Validate Income and Project Year Alignment

Confirm that the Income Layer covers only project years for which HMRC income data is available.

In [37]:
# Validate Income and Project Year alignment

income_years = set(income_features["Year"])
project_years = set([2018, 2019, 2020, 2021, 2022, 2023, 2024, 2025])

print("Income years:")
print(sorted(income_years))

print("\nProject years:")
print(sorted(project_years))

print("\nIncome years outside project period:")
print(sorted(income_years - project_years))

print("\nProject years without Income data:")
print(sorted(project_years - income_years))

print("\nShared years:")
print(sorted(income_years & project_years))

Income years:
[2018, 2019, 2020, 2021, 2022, 2023]

Project years:
[2018, 2019, 2020, 2021, 2022, 2023, 2024, 2025]

Income years outside project period:
[]

Project years without Income data:
[2024, 2025]

Shared years:
[2018, 2019, 2020, 2021, 2022, 2023]


In [38]:
from pathlib import Path

print(Path.cwd())

c:\Users\ASUS\projects\london-property-intelligence\notebooks


## Define Project Data Paths

Define the project-level processed data directory used to export the validated Income feature layer.

In [39]:
from pathlib import Path

# Project folders

PROJECT_ROOT = Path("..").resolve()
PROCESSED_DATA = PROJECT_ROOT / "data" / "processed"

print("Project root:", PROJECT_ROOT)
print("Processed data folder:", PROCESSED_DATA)

Project root: C:\Users\ASUS\projects\london-property-intelligence
Processed data folder: C:\Users\ASUS\projects\london-property-intelligence\data\processed


## Export Income Feature Layer

Save the validated Income feature table as an independent processed layer for future integration.

In [40]:
# Export validated Income feature layer

INCOME_OUTPUT_PATH = PROCESSED_DATA / "london_income.csv"

income_features.to_csv(
    INCOME_OUTPUT_PATH,
    index=False
)

print("Income feature layer saved successfully.")
print("Output path:", INCOME_OUTPUT_PATH)
print("Shape:", income_features.shape)

Income feature layer saved successfully.
Output path: C:\Users\ASUS\projects\london-property-intelligence\data\processed\london_income.csv
Shape: (198, 8)


## Validate Exported Income Layer

Reload the exported Income feature layer from disk and verify its structure, coverage and record count.

In [41]:
# Reload exported Income layer

income_exported = pd.read_csv(INCOME_OUTPUT_PATH)

print("=" * 60)
print("EXPORTED INCOME LAYER VALIDATION")
print("=" * 60)

print("Shape:", income_exported.shape)

print("\nColumns:")
print(income_exported.columns.tolist())

print("\nYears:")
print(sorted(income_exported["Year"].unique()))

print("\nUnique Boroughs:")
print(income_exported["Merge_Key"].nunique())

print("\nDuplicate Borough-Year records:")
print(
    income_exported.duplicated(
        subset=["Merge_Key", "Year"]
    ).sum()
)

print("\nMissing values:")
print(income_exported.isna().sum())

EXPORTED INCOME LAYER VALIDATION
Shape: (198, 8)

Columns:
['Code', 'Area', 'Tax_Year', 'Mean_Income', 'Median_Income', 'Number_of_Individuals', 'Year', 'Merge_Key']

Years:
[np.int64(2018), np.int64(2019), np.int64(2020), np.int64(2021), np.int64(2022), np.int64(2023)]

Unique Boroughs:
33

Duplicate Borough-Year records:
0

Missing values:
Code                     0
Area                     0
Tax_Year                 0
Mean_Income              0
Median_Income            0
Number_of_Individuals    0
Year                     0
Merge_Key                0
dtype: int64


## Income Layer — Final Status

The HMRC income dataset has been cleaned, validated and exported as an independent borough-year feature layer.

The final dataset contains 33 London boroughs across 6 project years (2018–2023), with complete coverage and no duplicate borough-year records.

The validated feature layer is saved as `london_income.csv` in the processed data directory and is ready for future integration into the London Intelligence Dataset.